In [1]:
import yaml
import pandas as pd
import requests

In [2]:
response=requests.get("https://datasets-server.huggingface.co/rows?dataset=cestwc%2Fbank-marketing&config=default&split=train&offset=0&length=100")

In [3]:
data_cols = response.json()

df_col_vals = pd.json_normalize(data_cols["features"])

df_col_vals = df_col_vals[["feature_idx", "name", "type.names"]]
df_col_vals

,feature_idx,name,type.names
0,0,age,NaN
1,1,job,"[admin., blue-collar, entrepreneur, housemaid,..."
2,2,marital,"[divorced, married, single, unknown]"
3,3,education,"[primary, secondary, tertiary, unknown, basic...."
4,4,default,"[no, yes, unknown]"
5,5,balance,NaN
6,6,housing,"[no, yes, unknown]"
7,7,loan,"[no, yes, unknown]"
8,8,contact,"[cellular, telephone, unknown]"
9,9,day,NaN


In [4]:
import requests
import pandas as pd

url = "https://huggingface.co/api/datasets/cestwc/bank-marketing/parquet/default/train"

response = requests.get(url)
files = response.json()

bf = pd.read_parquet(files[0])

In [5]:
bf

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,4,1,2,0,2143,1,0,2,5,4,261,1,999,0,3,0
1,44,9,2,1,0,29,1,0,2,5,4,151,1,999,0,3,0
2,33,2,1,1,0,2,1,1,2,5,4,76,1,999,0,3,0
3,47,1,1,3,0,1506,1,0,2,5,4,92,1,999,0,3,0
4,33,11,2,3,0,1,0,0,2,5,4,198,1,999,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,9,1,2,0,825,0,0,0,17,10,977,3,999,0,3,1
45207,71,5,0,0,0,1729,0,0,0,17,10,456,2,999,0,3,1
45208,72,5,1,1,0,5715,0,0,0,17,10,1127,5,184,3,2,1
45209,57,1,1,1,0,668,0,0,1,17,10,508,4,999,0,3,0


In [6]:
df_col_vals[df_col_vals['type.names'] != None]

,feature_idx,name,type.names
0,0,age,NaN
1,1,job,"[admin., blue-collar, entrepreneur, housemaid,..."
2,2,marital,"[divorced, married, single, unknown]"
3,3,education,"[primary, secondary, tertiary, unknown, basic...."
4,4,default,"[no, yes, unknown]"
5,5,balance,NaN
6,6,housing,"[no, yes, unknown]"
7,7,loan,"[no, yes, unknown]"
8,8,contact,"[cellular, telephone, unknown]"
9,9,day,NaN


In [7]:
df_col_vals = df_col_vals.dropna()

In [8]:
for _, row in df_col_vals.iterrows():
    col = row["name"]
    labels = row["type.names"]
    bf[col] = bf[col].map(dict(enumerate(labels)))

In [9]:
with open("../config.yaml",'r') as file:
    config = yaml.safe_load(file)

In [10]:
salaries = pd.read_csv(config['data']['clean']['file6'], quotechar='"', sep = ";")

In [11]:
salaries

,job,avg_salary_eur
0,unemployed,7091
1,entrepreneur,15000
2,unknown,11484
3,student,5742
4,retired,7697
5,admin.,12600
6,blue-collar,13000
7,housemaid,14400
8,management,16800
9,self-employed,15000


In [12]:
bf['income']=0

In [13]:
income_dict = salaries.set_index("job")["avg_salary_eur"].to_dict()
income_dict

{'unemployed': 7091,
 'entrepreneur': 15000,
 'unknown': 11484,
 'student': 5742,
 'retired': 7697,
 'admin.': 12600,
 'blue-collar': 13000,
 'housemaid': 14400,
 'management': 16800,
 'self-employed': 15000,
 'services': 13000,
 'technician': 11895}

In [14]:
bf["income"] = bf["job"].map(income_dict)

In [15]:
bf.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y', 'income'],
      dtype='str')

In [16]:
bf.isna().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
income       0
dtype: int64

In [17]:
bf.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
income       int64
dtype: object

In [18]:
bf.age.value_counts()
#bin later

age
32    2085
31    1996
33    1972
34    1930
35    1894
      ... 
95       2
92       2
93       2
88       2
94       1
Name: count, Length: 77, dtype: int64

In [19]:
bf.job.value_counts()

job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64

In [20]:
bf.marital.value_counts()

marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64

In [21]:
bf.default.value_counts()

default
no     44396
yes      815
Name: count, dtype: int64

In [22]:
bf.balance.value_counts()

balance
0        3514
1         195
2         156
4         139
3         134
         ... 
7038        1
9710        1
8205        1
14204       1
16353       1
Name: count, Length: 7168, dtype: int64

In [23]:
bf.housing.value_counts()

housing
yes    25130
no     20081
Name: count, dtype: int64

In [24]:
bf.loan.value_counts()

loan
no     37967
yes     7244
Name: count, dtype: int64

In [25]:
bf.contact.value_counts()

contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

In [26]:
bf.contact.value_counts()

contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

In [27]:
bf.day.value_counts()

day
20    2752
18    2308
21    2026
17    1939
6     1932
5     1910
14    1848
8     1842
28    1830
7     1817
19    1757
29    1745
15    1703
12    1603
13    1585
30    1566
9     1561
11    1479
4     1445
16    1415
2     1293
27    1121
3     1079
26    1035
23     939
22     905
25     840
31     643
10     524
24     447
1      322
Name: count, dtype: int64

In [28]:
bf.month.value_counts()

month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579
mar      477
dec      214
Name: count, dtype: int64

In [29]:
bf.duration.value_counts()
#this shouldn't be used in a predictive model, as it is not known before contacting the customer

duration
124     188
90      184
89      177
104     175
114     175
       ... 
1440      1
1405      1
1298      1
1246      1
1556      1
Name: count, Length: 1573, dtype: int64

In [30]:
bf.campaign.value_counts()
#needs to be binned

campaign
1     17544
2     12505
3      5521
4      3522
5      1764
6      1291
7       735
8       540
9       327
10      266
11      201
12      155
13      133
14       93
15       84
16       79
17       69
18       51
19       44
20       43
21       35
22       23
25       22
23       22
24       20
28       16
29       16
26       13
31       12
27       10
32        9
30        8
33        6
34        5
35        4
36        4
43        3
38        3
41        2
50        2
37        2
51        1
63        1
55        1
46        1
58        1
39        1
44        1
Name: count, dtype: int64

In [31]:
bf.pdays.value_counts(), bf.pdays.nunique()
#-1 means never been contacted

(pdays
 999    36954
 182      167
 92       147
 91       126
 183      126
        ...  
 541        1
 543        1
 871        1
 550        1
 530        1
 Name: count, Length: 559, dtype: int64,
 559)

In [32]:
bf['previously_contacted']= bf.pdays.apply([lambda x: 0 if x==-1 else 1])

In [33]:
bf['previously_contacted'].value_counts()

previously_contacted
1    45211
Name: count, dtype: int64

In [34]:
bf = bf.drop(columns='previously_contacted')

In [35]:
bf.pdays.max()

np.int64(999)

In [36]:
bf.previous.value_counts()
#needs to be binned

previous
0      36954
1       2772
2       2106
3       1142
4        714
5        459
6        277
7        205
8        129
9         92
10        67
11        65
12        44
13        38
15        20
14        19
17        15
16        13
19        11
20         8
23         8
18         6
22         6
27         5
24         5
21         4
25         4
29         4
30         3
26         2
37         2
38         2
28         2
51         1
275        1
58         1
32         1
40         1
55         1
35         1
41         1
Name: count, dtype: int64

In [37]:
bf.poutcome.value_counts()

poutcome
unknown    36959
failure     4901
other       1840
success     1511
Name: count, dtype: int64

In [38]:
bf.y.value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [39]:
bf.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y', 'income'],
      dtype='str')

In [40]:
bf.to_csv("../data/clean/bank-full.csv", index=False, encoding= "utf-8", sep = ";")